[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/07_%E9%A4%90%E9%A3%B2_%E9%96%80%E5%B8%82%E6%AF%8F%E6%97%A5%E7%87%9F%E6%94%B6%E9%A0%90%E4%BC%B0/%E9%96%80%E5%B8%82%E6%AF%8F%E6%97%A5%E7%87%9F%E6%94%B6%E9%A0%90%E4%BC%B0_%E8%BF%B4%E6%AD%B8%E7%AF%87.ipynb)

# 🍴 範例 07：餐飲連鎖—門市每日營收預估與智慧排班 (迴歸篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「連鎖餐廳店長、餐飲品牌營運主管與餐飲創業者（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以門市經營白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、餐飲門市資料集介紹 (`store_revenue.csv`)

本數據集記錄了餐飲門市歷年 1,100 個營業日的天氣、商圈客流與最終營業額成效：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 餐飲門市業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **商圈型態** | 類別型 (`object`) | 信義商務區、西門觀光區、板橋住宅區、內科園區 | 門市客群結構 |
| **星期別** | 類別型 (`object`) | 星期一 到 星期日 | 週間客流波動週期 |
| **是否為國定假日** | 類別型 (`object`) | 是、否 | 國定假日與連續假期加成 |
| **天氣狀況** | 類別型 (`object`) | 晴朗多雲、晴朗炎熱、局部陣雨、大雨滂沱 | 天候對聚餐外出動機之影響 |
| **最高氣溫_度C** | 數值型 (`float64`) | 28.0 (度) | 氣候溫度變化 |
| **商圈活動熱度** | 數值型 (`int64`) | 80 (分) | 週邊展演大活動指數 |
| **當日訂位客席率** | 數值型 (`float64`) | 0.85 (即 85% 已被預訂) | 開店前已知之保底客席率 |
| **每日營收_TWD (目標 y)** | 數值型 (`int64`) | **245,000 (元)** | **預測目標：當日最終結帳營業額（連續數值迴歸）** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視天候與星期別之營收分佈。
3. **特徵相關性與視覺化**：繪製客席訂位率 vs 營收散佈圖與氣溫箱型圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「線性迴歸」與「隨機森林迴歸器」。
6. **測試集大考與門市備料策略**：評估 R²、MAE，輸出模型並模擬次日預估。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入餐飲門市營收預測分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並確保門市營運分析圖表中的中文字型與座標軸負號皆能正常繪製。

---
💡 **AI 提問小抄：**

1. 📌 **外部天候特徵之價值**
   > 「為什麼在餐飲門市營收預測中，加入『天氣預報』與『降雨機率』能大幅提升預測精準度？大雨天對內用餐廳與外帶飲料店的營收影響方向有一樣嗎？」


---  
### 【步驟 1】：載入資料與餐飲營運探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 store_revenue.csv 並檢視資料概況
df = pd.read_csv("store_revenue.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 營收數值統計摘要 ---")
display(df.describe().round(0))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  檢查 1,100 個營業日的數據概況，確認門市營收平均約 17 萬元、氣溫平均 26 度。

---
💡 **AI 提問小抄：**

1. 📌 **營收極端值檢查**
   > 「在統計摘要中，最高日營收達 32 萬元，最低只有 4.5 萬元。這些極端高峰（如跨年夜）與低谷（如颱風天停班）在模型訓練時應該保留還是當成異常值剔除？」


---  
### 【步驟 2】：天氣狀況與商圈型態之營收交叉圖表繪製

In [ ]:
# [步驟 2 程式碼] 繪製天氣狀況與星期別的營業額分佈圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="天氣狀況", y="每日營收_TWD", palette="Blues_r", ax=axes[0])
axes[0].set_title("不同天氣狀況之每日營收分佈")
axes[0].set_ylabel("每日營收 (TWD)")

week_order = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六", "星期日"]
sns.barplot(data=df, x="星期別", y="每日營收_TWD", order=week_order, palette="YlOrBr", ax=axes[1])
axes[1].set_title("一週七天平均營收表現")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  繪製箱型圖與長條圖，直觀呈現大雨滂沱對營收的壓抑效果，以及週五與週末兩天的營收爆發力。

---
💡 **AI 提問小抄：**

1. 📌 **週期性（Seasonality）特徵工程**
   > 「星期六日的營收通常是平日的 1.5 倍。除了將『星期別』作為文字類別處理外，在時序分析中還有哪些將月份、發薪日納入考慮的高級特徵工程手法？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["每日營收_TWD"])
y = df["每日營收_TWD"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["最高氣溫_度C", "商圈活動熱度", "當日訂位客席率"]
cat_cols = ["商圈型態", "星期別", "是否為國定假日", "天氣狀況"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 天，測試集：{X_test.shape[0]} 天")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  拆分 80% 訓練集與 20% 測試集，將氣溫與訂位率標準化，商圈與天候進行獨熱編碼打包。

---
💡 **AI 提問小抄：**

1. 📌 **防偷看答案與資料洩漏**
   > 「在餐飲實務中，如果我們不小心把『當日總開立發票金額』放進 X 矩陣中，模型測試集 R² 會高達 0.9999。為什麼這種看似神準的模型在明天開店前完全無用？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (線性迴歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證評估線性迴歸與隨機森林
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
pipe_rf = Pipeline([("preprocessor", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42))])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="r2")

print(f"線性迴歸 5 折交叉驗證 平均 R²: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 R²: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  運用 5 折交叉驗證，客觀比對兩種演算法對天候與商圈非線性互動的擬合能力。

---
💡 **AI 提問小抄：**

1. 📌 **非線性天候綜效**
   > 「『假日遇上下大雨』跟『平日遇上下大雨』對百貨商場店的影響截然不同（假日大雨反而湧入百貨避雨用餐）。隨機森林是如何透過多層決策樹捕捉這種交互綜效？」


---  
### 【步驟 5】：測試集大考與指標解讀 (MAE, RMSE, R²)

In [ ]:
# [步驟 5 程式碼] 在全新測試集進行評估並繪製預測散佈圖
from sklearn.metrics import r2_score, mean_absolute_error

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"=== 測試集最終評估結果 ===")
print(f"決定係數 R²: {r2:.4f} (模型解釋了 {r2*100:.1f}% 的門市營收變異)")
print(f"平均絕對誤差 MAE: ${mae:,.0f} 元 (平均每日預測誤差金額)")

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color="#b45309")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="完美預測對角線")
plt.title(f"門市真實營收 vs 模型預測營收散佈圖 (R²={r2:.2f})")
plt.xlabel("真實營業額 (TWD)")
plt.ylabel("模型預測營業額 (TWD)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將模型應用於 220 個從未見過的測試營業日，評估 MAE 誤差金額與 $R^2$ 解釋力。

---
💡 **AI 提問小抄：**

1. 📌 **MAE 與兼職時薪的換算**
   > 「如果我們的 MAE 平均誤差約為 8,000 元，相當於 15 位客人的營業額。店長在根據這個模型排班時，應該保留幾位『機動彈性兼職 (On-call PT)』以兼顧人事成本與服務品質？」


---  
### 【步驟 6】：次日營收即時預估 (Inference) 與生鮮備料排班

In [ ]:
# [步驟 6 程式碼] 保存模型管線並示範次日門市營運即時試算
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬店長今晚準備排定明天「星期六」信義店的營運方案
tomorrow = pd.DataFrame([{
    "商圈型態": "信義商務區",
    "星期別": "星期六",
    "是否為國定假日": "是",
    "天氣狀況": "晴朗多雲",
    "最高氣溫_度C": 29.0,
    "商圈活動熱度": 85,
    "當日訂位客席率": 0.90
}])

loaded_model = joblib.load("model.joblib")
pred_rev = loaded_model.predict(tomorrow)[0]

print(f"\n📢 次日門市營運預估報告：")
print(f"預估明日營業額：${pred_rev:,.0f} 元")
print(f"建議內外場排班：預期滿座翻桌，建議排班 8 位外場（含 3 名 PT）與 5 位內場廚師備料！")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  保存管線檔案，並模擬店長在打烊前輸入明天的天氣與預約率，自動產生營收預測與排班指引。

---
💡 **AI 提問小抄：**

1. 📌 **自動化氣象 API 串接**
   > 「在連鎖餐飲總部中，我們該如何使用 Python 自動爬取中央氣象署的 3 天天氣預報 API，自動餵入這個模型，每晚定時產出全台 50 家分店的次日備料建議清單？」
